In [1]:
import plotly.graph_objects as go
import models
import diagnostics
import os
import preprocess
import plots
import utils
import numpy as np
import pandas as pd

### Setup

In [2]:
file_path = os.getenv("FILE_PATH")
SNP_file = './data/S&P 500 Index.csv'
VIX_file = './data/S&P 500 VIX.csv'

### S&P500

In [3]:
SNP_processed = preprocess.preprocess_data(SNP_file, 'price', 'log_returns')
SNP_processed.head()

,price,log_returns
date,,
2015-01-05,2020.58,-0.018447
2015-01-06,2002.61,-0.008933
2015-01-07,2025.90,0.011563
2015-01-08,2062.14,0.017730
2015-01-09,2044.81,-0.008439


In [4]:
snp_obj = utils.FbmReturnForecast(SNP_processed)
snp_obj.df.head()

,price,log_returns
date,,
2015-01-05,2020.58,-0.018447
2015-01-06,2002.61,-0.008933
2015-01-07,2025.90,0.011563
2015-01-08,2062.14,0.017730
2015-01-09,2044.81,-0.008439


In [5]:
snp_fbm = utils.apply_rolling_predictions_from_start(snp_obj, '2017-01-01', 250)
snp_fbm.head()

Day 252...
Day 253...
Day 254...
Day 255...
Day 256...
Day 257...
Day 258...
Day 259...
Day 260...
Day 261...
Day 262...
Day 263...
Day 264...
Day 265...
Day 266...
Day 267...
Day 268...
Day 269...
Day 270...
Day 271...
Day 272...
Day 273...
Day 274...
Day 275...
Day 276...
Day 277...
Day 278...
Day 279...
Day 280...
Day 281...
Day 282...
Day 283...
Day 284...
Day 285...
Day 286...
Day 287...
Day 288...
Day 289...
Day 290...
Day 291...
Day 292...
Day 293...
Day 294...
Day 295...
Day 296...
Day 297...
Day 298...
Day 299...
Day 300...
Day 301...
Day 302...
Day 303...
Day 304...
Day 305...
Day 306...
Day 307...
Day 308...
Day 309...
Day 310...
Day 311...
Day 312...
Day 313...
Day 314...
Day 315...
Day 316...
Day 317...
Day 318...
Day 319...
Day 320...
Day 321...
Day 322...
Day 323...
Day 324...
Day 325...
Day 326...
Day 327...
Day 328...
Day 329...
Day 330...
Day 331...
Day 332...
Day 333...
Day 334...
Day 335...
Day 336...
Day 337...
Day 338...
Day 339...
Day 340...
Day 341...
Day 342...

,predicted,conditional_vol
2016-01-05,2012.409126,0.009764
2016-01-06,2016.794784,0.009699
2016-01-07,1990.939472,0.009720
2016-01-08,1942.634077,0.009824
2016-01-11,1923.101152,0.009847


In [6]:
snp_fbm= utils.compute_log_returns(
  snp_fbm,
  SNP_processed,
  "predicted",
  "predicted_log_returns"
)
snp_fbm.head()

,predicted,conditional_vol,predicted_log_returns,price,log_returns
2016-01-06,2016.794784,0.009699,0.002177,1990.26,-0.013202
2016-01-07,1990.939472,0.009720,-0.012903,1943.09,-0.023986
2016-01-08,1942.634077,0.009824,-0.024562,1922.03,-0.010898
2016-01-11,1923.101152,0.009847,-0.010106,1923.67,0.000853
2016-01-12,1923.292735,0.009840,0.000100,1938.68,0.007773


In [7]:
diagnostics.in_sample_diagnostics(snp_fbm['predicted_log_returns'], snp_fbm['log_returns'], snp_fbm['conditional_vol'])

Jarque-Bera test p-value: 0.00000
Ljung-Box (residuals) p-value, 0.00000
Ljung-Box (residuals^2) p-value, 0.00000


In [8]:
snp_fbm_var = snp_fbm
snp_fbm_var['VaR_99'] = snp_fbm_var['predicted_log_returns'].rolling(250).quantile(0.01)
diagnostics.compute_var_violations(snp_fbm_var, 'VaR_99', 'predicted_log_returns')

{'actual_exceedances': 33,
 'expected_exceedances': 20.12000000000002,
 'violation_ratio': 1.6401590457256445}

In [9]:
utils.compute_rmse(snp_fbm_var, 'log_returns', 'predicted_log_returns')

0.017349333189397246

In [10]:
diagnostics.bernoulli_coverage_test(snp_fbm_var, var_col='VaR_99', predicted_col='predicted_log_returns')

(0.008243400511761534, 6.97981770732531)

In [11]:
diagnostics.compute_hit_rate(predicted=snp_fbm_var['predicted_log_returns'], actual=snp_fbm_var['log_returns'])


Hit Rate: 48.39%


0.4838567005749668

In [12]:
plots.plot_var_violations(snp_fbm_var, var_col='VaR_99', predicted_col='predicted_log_returns')

In [13]:
plots.actual_vs_predicted_time_series_plot(
  snp_fbm_var,
  'log_returns',
  'predicted_log_returns',
  'Log Returns'
)

In [24]:
from hurst import compute_Hc
import plotly.graph_objects as go

def compute_rolling_hurst(df, window, col='log_returns', lag=0):
    hurst_values = []
    times = []
    
    # Loop through the DataFrame using the rolling window
    for i in range(window - 1, len(df), lag+1):

        # Extract the window slice from the series
        window_series = df[col].iloc[i - window + 1 : i + 1: lag+1]
        
        # Compute the Hurst exponent using the 'change' method and simplified calculation
        h, c, data = compute_Hc(window_series, kind='change', simplified=True)
        
        hurst_values.append(h)
        times.append(df.index[i])
    
    # Create and return a new DataFrame with the computed Hurst exponents
    result_df = pd.DataFrame({'hurst': hurst_values}, index=times)
    return result_df

hurst_window250_lag0 = compute_rolling_hurst(snp_fbm_var, window=250, col='log_returns', lag=0)
hurst_window350_lag0 = compute_rolling_hurst(snp_fbm_var, window=350, col='log_returns', lag=0)
hurst_window500_lag0 = compute_rolling_hurst(snp_fbm_var, window=500, col='log_returns', lag=0)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=hurst_window250_lag0.index,
    y=hurst_window250_lag0["hurst"],
    mode='lines',
    name='250 days rolling window',
    line=dict(color='blue')
))
fig.add_trace(go.Scatter(
    x=hurst_window350_lag0.index,
    y=hurst_window350_lag0["hurst"],
    mode='lines',
    name='350 days rolling window',
    line=dict(color='red')
))
fig.add_trace(go.Scatter(
    x=hurst_window500_lag0.index,
    y=hurst_window500_lag0["hurst"],
    mode='lines',
    name='500 days rolling window',
    line=dict(color='green')
))
fig.update_layout(
    title=f"Hurst exponent for SNP",
    xaxis_title="Date",
    yaxis_title="Hurst Value",
    template="plotly_white"
)

### VIX

In [14]:
VIX_processed = preprocess.preprocess_data(VIX_file, "vol", "log_vol_diff")
VIX_processed.head()

,vol,log_vol_diff
date,,
2015-01-05,19.92,0.113088
2015-01-06,21.12,0.058496
2015-01-07,19.31,-0.089597
2015-01-08,17.01,-0.126822
2015-01-09,17.55,0.031253


In [15]:
vix_obj = utils.FbmVolForecast(VIX_processed)
vix_fbm = utils.apply_rolling_predictions_from_start(vix_obj, '2017-01-01', 250)

Day 252...
Day 253...
Day 254...
Day 255...
Day 256...
Day 257...
Day 258...
Day 259...
Day 260...
Day 261...
Day 262...
Day 263...
Day 264...
Day 265...
Day 266...
Day 267...
Day 268...
Day 269...
Day 270...
Day 271...
Day 272...
Day 273...
Day 274...
Day 275...
Day 276...
Day 277...
Day 278...
Day 279...
Day 280...
Day 281...
Day 282...
Day 283...
Day 284...
Day 285...
Day 286...
Day 287...
Day 288...
Day 289...
Day 290...
Day 291...
Day 292...
Day 293...
Day 294...
Day 295...
Day 296...
Day 297...
Day 298...
Day 299...
Day 300...
Day 301...
Day 302...
Day 303...
Day 304...
Day 305...
Day 306...
Day 307...
Day 308...
Day 309...
Day 310...
Day 311...
Day 312...
Day 313...
Day 314...
Day 315...
Day 316...
Day 317...
Day 318...
Day 319...
Day 320...
Day 321...
Day 322...
Day 323...
Day 324...
Day 325...
Day 326...
Day 327...
Day 328...
Day 329...
Day 330...
Day 331...
Day 332...
Day 333...
Day 334...
Day 335...
Day 336...
Day 337...
Day 338...
Day 339...
Day 340...
Day 341...
Day 342...

In [16]:
vix_fbm= utils.compute_log_returns(
  vix_fbm,
  VIX_processed,
  "predicted",
  "predicted_log_vol_diff"
)
vix_fbm.head()

,predicted,conditional_vol,predicted_log_vol_diff,vol,log_vol_diff
2016-01-06,19.432059,0.086427,-0.067902,20.59,0.062630
2016-01-07,20.663427,0.086494,0.061441,24.99,0.193670
2016-01-08,25.116313,0.087079,0.195152,27.01,0.077731
2016-01-11,27.174703,0.087165,0.078769,24.30,-0.105731
2016-01-12,24.384898,0.087384,-0.108322,22.47,-0.078295


In [17]:
diagnostics.in_sample_diagnostics(vix_fbm['predicted_log_vol_diff'], vix_fbm['log_vol_diff'], vix_fbm['conditional_vol'])

Jarque-Bera test p-value: 0.00000
Ljung-Box (residuals) p-value, 0.00000
Ljung-Box (residuals^2) p-value, 0.00000


In [18]:
vix_fbm_var = vix_fbm
vix_fbm_var['VaR_99'] = vix_fbm_var['predicted_log_vol_diff'].rolling(250).quantile(0.01)  # 1st percentile
diagnostics.compute_var_violations(vix_fbm_var, 'VaR_99', 'predicted_log_vol_diff')

{'actual_exceedances': 30,
 'expected_exceedances': 20.33000000000002,
 'violation_ratio': 1.4756517461878982}

In [19]:
utils.compute_rmse(vix_fbm_var, 'log_vol_diff', 'predicted_log_vol_diff')

0.1160381907265143

In [20]:
diagnostics.bernoulli_coverage_test(vix_fbm_var, var_col='VaR_99', predicted_col='predicted_log_vol_diff')

(0.044105459765187094, 4.052519952552018)

In [21]:
diagnostics.compute_hit_rate(predicted=vix_fbm_var['predicted_log_vol_diff'], actual=vix_fbm_var['log_vol_diff'])


Hit Rate: 49.30%


0.492988606485539

In [22]:
plots.plot_var_violations(vix_fbm_var, var_col='VaR_99', predicted_col='predicted_log_vol_diff')

In [23]:
plots.actual_vs_predicted_time_series_plot(
  vix_fbm_var,
  'log_vol_diff',
  'predicted_log_vol_diff',
  'Log Returns on Volatility'
)